# ForecastEx

## Web REST API

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests, json, os
from pprint import pprint

## Get live markets

In [3]:
from get_live_markets import get_live_markets
markets = get_live_markets()

In [35]:
from add_category_to_markets import add_category_to_markets
df_markets = add_category_to_markets(markets)

In [36]:
df_markets = df_markets[['category', 'name', 'symbol', 'conid']].sort_values(by=['category', 'name'])

In [37]:
conid_to_market = {market['conid']: market for market in markets}

In [38]:
df_markets.head(3)

,category,name,symbol,conid
126,Climate,Annual Global Temperature Threshold,GTTA,776339242
82,Climate,Atmospheric Carbon Dioxide,ACD,712856705
84,Climate,Global Carbon Dioxide Emissions,GCE,732764729


## Get forecastex markets excluding financials

In [11]:
from fetch_all_contracts import fetch_all_contracts
from tqdm.auto import tqdm

In [12]:
for market in tqdm(markets):
    if 'contracts' not in market:
        market['contracts'] = fetch_all_contracts(market['conid'])
        print(market['conid'], len(market['contracts']))

  0%|          | 0/206 [00:00<?, ?it/s]

796056051 10
658663572 0
791099755 30
766914439 8
712856730 0
745923947 0
752108858 0
766914639 4
788713854 44
788713829 0
788713839 0
796079989 0
796079999 0
796080005 0
796080015 0
796080020 0
796080045 0
798491013 30
798491023 30
796080030 6
796080039 10
793085633 0
793085658 0
793085673 0
793085682 0
793085688 0
786030226 0
786030232 0
786030241 0
786554238 0
786554271 0
786554284 0
771326556 4
766914564 4
767906110 8
766914574 6
766914581 2
800591677 30
725452870 0
733131966 4
745923952 4
796056057 2
800591710 16
573031117 0
582530257 0
626425574 0
751047555 0
771326535 8
791099721 0
791099730 0
791099793 0
791099803 0
791099770 0
791099776 0
791099786 0
791099827 0
791099833 0
802185942 0
802185948 0
767906100 4
767906124 2
626425591 0
658663567 0
712856689 0
712856715 0
726203920 0
771326527 2
745923962 12
745923968 12
745923978 4
725452876 0
726203945 0
766914502 4
800320229 0
726203939 0
798087648 30
766914423 4
766914429 6
799576107 0
768815597 4
771326565 10
766914591 4
7128

In [13]:
forecastex = [market for market in markets if len(market['contracts']) > 0]

In [14]:
len(markets), len(forecastex)

(206, 65)

## Get current probability for each market contract

In [15]:
from get_candidate_probability import get_candidate_probability

In [16]:
from tqdm.auto import tqdm

In [17]:
for market in tqdm(forecastex):
    for contract in market['contracts']:
        conid = contract['conid']
        if 'probability' not in contract:
            try:
                contract['probability'] = get_candidate_probability(conid)
            except:
                contract['probability'] = None
                print('error', contract['shortDescription'])
        try:
            print(conid, contract['shortDescription'], contract['probability']['probability_pct'])
        except:
            pass

  0%|          | 0/65 [00:00<?, ?it/s]

796056496 MNYCG Nov04'25 Sliwa YES @FORECASTX 6.0
796056501 MNYCG Nov04'25 Sliwa NO @FORECASTX 95.0
796056506 MNYCG Nov04'25 Cuomo YES @FORECASTX 16.0
796056511 MNYCG Nov04'25 Cuomo NO @FORECASTX 85.0
796056520 MNYCG Nov04'25 Mamdani YES @FORECASTX 71.0
796056525 MNYCG Nov04'25 Mamdani NO @FORECASTX 30.0
796056531 MNYCG Nov04'25 Adams YES @FORECASTX 9.0
796056534 MNYCG Nov04'25 Adams NO @FORECASTX 92.0
791103640 FFDEC Jul30'25 Lower25bps YES @FORECASTX 9.0
791103645 FFDEC Jul30'25 Lower25bps NO @FORECASTX 92.0
791103649 FFDEC Jul30'25 Nochange YES @FORECASTX 88.5
791103652 FFDEC Jul30'25 Nochange NO @FORECASTX 12.5
791103682 FFDEC Sep17'25 Lower25bps YES @FORECASTX 59.0
791103687 FFDEC Sep17'25 Lower25bps NO @FORECASTX 42.0
791103688 FFDEC Sep17'25 Nochange YES @FORECASTX 45.0
791103693 FFDEC Sep17'25 Nochange NO @FORECASTX 56.00000000000001
791103699 FFDEC Sep17'25 Raise50bpsormore YES @FORECASTX 11.0
791103702 FFDEC Sep17'25 Raise50bpsormore NO @FORECASTX 90.0
791103705 FFDEC Sep17'2

In [18]:
f2 = [x.copy() for x in forecastex]

In [19]:
for market in f2:
    market['contracts'] = [x for x in market['contracts'] if x['probability']]

In [20]:
f2 = [market for market in f2 if market['contracts']]

In [21]:
len(f2), len(forecastex)

(26, 65)

## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [22]:
from OI_to_integer import OI_to_integer

In [24]:
from run_get_OI import run_get_OI

In [ ]:
for market in tqdm(f2):
    for contract in tqdm(market['contracts']):
        if 'OI' not in contract:
            conid = contract['conid']
            OI = await run_get_OI([conid])
            OI = list(OI.values())[0]
            contract['OI'] = OI_to_integer(OI)

In [33]:
contract

{'market': 'California',
 'popularityRank': 1032271,
 'name': "GPCAD Jun02'26 Harris",
 'longDescription': 'Will Kamala Harris win the California Democratic primary for governor in 2026?',
 'putOrCall': 'P',
 'expiration': '20260717',
 'currency': 'USD',
 'lastTradeMillis': 1784332740000,
 'lastTradeDate': '20260717',
 'lastTradeTime': '1859',
 'timezone': 'America/Chicago',
 'commodityCode': 'GPCAD',
 'eventAuthorityURL': 'https://www.sos.ca.gov/elections',
 'eventFixedPayout': '1',
 'sourceAgency': 'California Secretary of State Elections Division',
 'marketRulesLink': 'https://data.forecastex.com/regulatory/GPTermsandConditions.pdf',
 'underlyingName': 'California Governor Democratic Primary',
 'categories': ['g17493', 'g17467', 'g7428'],
 'expectedResolutionTime': '20260717185959',
 'expectedPayoutTime': '20260718130000',
 'timespecifierParam': '2026.6.2',
 'exchange': 'FORECASTX',
 'priceIncrement': 0.01,
 'tradingHours': {'timezone': 'America/Chicago', 'holidays': []},
 'conid': 

## Collect and filter

In [ ]:
rows = []
for market in f2:
    for ctx in market['contracts']:
        contract = ctx.copy()
        probability = contract['probability']
        contract['probability_pct'] = probability['probability_pct']
        contract['weekly_volume'] = sum(probability['volume'])
        contract['yesNo'] = 'YES' if contract['putOrCall'] == 'C' else 'NO'
        for x in ['putOrCall','popularityRank', 'expiration', 'lastTradeMillis', 'lastTradeTime','eventFixedPayout', 'commodityCode', 'strike', 'market',
                    'categories', 'expectedResolutionTime', 'expectedPayoutTime', 'timespecifierParam', 'sourceAgency', 'marketRulesLink', 
                    'exchange', 'priceIncrement', 'currency', 'timezone', 'eventAuthorityURL', 'probability', 'tradingHours']:
            del contract[x]
        rows.append(contract)

In [39]:
from add_category_to_markets import add_category_to_markets
df = add_category_to_markets(rows, 'longDescription')

In [40]:
df = df[['category','underlyingName', 'longDescription', 'shortDescription', 'strikeLabel', 'yesNo', 'probability_pct',  'OI', 'lastTradeDate', 'conid']].sort_values(by=['category', 'underlyingName', 'strikeLabel', 'yesNo']).reset_index(drop=True)
df1 = df[(df.probability_pct > 25 ) & (df.probability_pct < 75 ) & (df.yesNo == 'YES')].reset_index(drop=True)

In [41]:
df1

,category,underlyingName,longDescription,shortDescription,strikeLabel,yesNo,probability_pct,OI,lastTradeDate,conid
0,Election,California Governor Democratic Primary,Will Kamala Harris win the California Democratic primary for governor in 2026?,GPCAD Jun02'26 Harris YES @FORECASTX,Harris,YES,31.5,140.0,20260717,767285037
1,Election,Florida Governor Republican Primary,Will Byron Donalds win the Florida Republican primary for governor in 2026?,GPFLR Aug18'26 Donalds YES @FORECASTX,Donalds,YES,73.0,288.0,20260903,767285117
2,Election,General Election for New York City Mayor,Will Zohran Mamdani win the New York City general election for mayor in 2025?,MNYCG Nov04'25 Mamdani YES @FORECASTX,Mamdani,YES,71.0,1190000.0,20251129,796056520
3,Election,Kansas Governor Democratic Primary,Will Ethan Corson win the Kansas Democratic primary for governor in 2026?,GPKSD Aug04'26 Corson YES @FORECASTX,Corson,YES,36.0,250.0,20260911,767285268
4,Election,Kansas Governor Democratic Primary,Will Julie Holscher win the Kansas Democratic primary for governor in 2026?,GPKSD Aug04'26 Holscher YES @FORECASTX,Holscher,YES,39.0,290.0,20260911,767285263
5,Election,Mayoral Democratic Primary in Seattle,Will Bruce Harrell win the Seattle Democratic Primary election for Mayor in 2025?,MSEAD Aug05'25 Harrell YES @FORECASTX,Harrell,YES,61.0,1680.0,20250819,771327516
6,Election,Mayoral General Election in Minneapolis,Will Omar Fateh win the Minneapolis general election for Mayor in 2025?,MMSPG Nov04'25 Fateh YES @FORECASTX,Fateh,YES,40.0,304.0,20251114,771327431
7,Election,Mayoral General Election in Minneapolis,Will Jacob Frey win the Minneapolis general election for Mayor in 2025?,MMSPG Nov04'25 Frey YES @FORECASTX,Frey,YES,51.0,170.0,20251114,771327413
8,Election,New Mexico Governor Democratic Primary,Will Deb Haaland win the New Mexico Democratic primary for governor in 2026?,GPNMD Jun02'26 Haaland YES @FORECASTX,Haaland,YES,71.0,405.0,20260630,767285535
9,Election,New York Governor Democratic Primary,Will Kathy Hochul win the New York Democratic primary for governor in 2026?,GPNYD Jun23'26 Hochul YES @FORECASTX,Hochul,YES,71.0,124.0,20260730,767285565


In [42]:
df1.to_csv('forecastex.csv')

In [43]:
df2 = df1.sort_values(by=['underlyingName', 'strikeLabel'])
df2 = df2[['underlyingName', 'longDescription', 'shortDescription', 'strikeLabel', 'yesNo', 'probability_pct', 'OI', 'lastTradeDate', 'conid']]
df2 = df2[df2.lastTradeDate <= '20251028']
df2.sort_values(by='lastTradeDate')

,underlyingName,longDescription,shortDescription,strikeLabel,yesNo,probability_pct,OI,lastTradeDate,conid
5,Mayoral Democratic Primary in Seattle,Will Bruce Harrell win the Seattle Democratic Primary election for Mayor in 2025?,MSEAD Aug05'25 Harrell YES @FORECASTX,Harrell,YES,61.0,1680.0,20250819,771327516
16,Fed Decision,"Will the Fed lower the rate 25 bps at the September 17, 2025 meeting?",FFDEC Sep17'25 Lower25bps YES @FORECASTX,Lower 25bps,YES,59.0,450.0,20250917,791103682
18,Fed Decision,"Will the Fed leave the rate unchanged at the September 17, 2025 meeting?",FFDEC Sep17'25 Nochange YES @FORECASTX,No change,YES,45.0,481.0,20250917,791103688
21,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Automobiles and Auto Parts by September 30, 2025?",USTRF Sep30'25 Automobiles YES @FORECASTX,Automobiles,YES,30.0,10.0,20251001,788715383
23,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on the European Union by September 30, 2025?",USTRF Sep30'25 EuropeanUnion YES @FORECASTX,EuropeanUnion,YES,32.0,20.0,20251001,788715389
24,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Russia by September 30, 2025?",USTRF Sep30'25 Russia YES @FORECASTX,Russia,YES,45.0,30.0,20251001,788715372
25,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Semiconductors by September 30, 2025?",USTRF Sep30'25 Semiconductors YES @FORECASTX,Semiconductors,YES,37.0,70.0,20251001,788715405
27,US Tariffs Enacted via Executive Order,"Will the United States impose additional tariffs on Smartphones by September 30, 2025?",USTRF Sep30'25 Smartphones YES @FORECASTX,Smartphones,YES,38.0,20.0,20251001,788715420


In [44]:
from datetime import datetime

In [46]:
from make_glimt_question import make_glimt_question

In [47]:
ifps = df2.apply(make_glimt_question, axis=1)

In [48]:
len(ifps)

8

In [49]:
from save_ifps_to_disk import save_ifps_to_disk
id_to_ifp = save_ifps_to_disk(ifps)

In [50]:
from gather_news_for_ifps import gather_news_for_ifps
news = gather_news_for_ifps(ifps)

saved glimt/news/791103682.txt
saved glimt/news/791103688.txt
saved glimt/news/771327516.txt
saved glimt/news/788715383.txt
saved glimt/news/788715389.txt
saved glimt/news/788715372.txt
saved glimt/news/788715405.txt
saved glimt/news/788715420.txt


In [227]:
from detailed_proposition import detailed_proposition
from wiki_semantic_search import wiki_semantic_search
from split_news_into_text_and_urls import split_news_into_text_and_urls
from create_source_summaries import create_source_summaries
from rephrase_binary_outcomes import rephrase_binary_outcomes
from format_research import format_research
from glimt_forecast_prompt import glimt_forecast_prompt
from humor_me import humor_me
from get_forecast_components import *
from median_forecast import median_forecast
from median_rationale import median_rationale
from datetime import datetime

loading massive wiki index 2025-07-24 21:51:33.210260
loading wiki article titles 2025-07-24 21:52:41.002267
loading sentence transformer model 2025-07-24 21:52:44.181085
done 2025-07-24 21:52:49.595186


In [ ]:
for ifp in ifps[1:]:
    print('begin FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())
    title_plus_criteria = detailed_proposition(ifp)
    wiki_articles = wiki_semantic_search(title_plus_criteria)
    ifp_news_sources, ifp_news_text = split_news_into_text_and_urls(ifp, news)
    source_summaries, sources = create_source_summaries(ifp['id'], title_plus_criteria, wiki_articles, ifp_news_sources, ifp_news_text)
    rephrase_binary_outcomes(ifp)
    research = format_research(source_summaries)
    prompt, rejected = glimt_forecast_prompt(ifp, research)
    ## Run the prompt 5 times
    prompt_tries = 2 # Waste of time on Mistral 4 bit
    answers = [humor_me(prompt, i+1) for i in range(prompt_tries)]
    binProbs = [get_bin_probs(a) for a in answers]
    rights = [get_rights(a) for a in answers]
    wrongs = [get_wrongs(a) for a in answers]
    ## Median forecasts and rationales
    forecast = rejected + median_forecast(binProbs)
    right = median_rationale(rights)
    wrong = median_rationale(wrongs)
    result = (forecast, right, wrong, sources)
    fn = f'glimt/forecast'
    import os
    os.makedirs(fn, exist_ok=True)
    fn = f"{fn}/{ifp['id']}.json"
    import json
    with open(fn, 'w') as f:
        json.dump(result, f)


In [253]:
ifp

{'id': 788715420,
 'type': 'bins',
 'state': 'active',
 'dates': {'startDay': '20250724', 'endDay': '20251001'},
 'props': {'title': 'Will the United States impose additional tariffs on Smartphones by September 30, 2025?',
  'shortTitle': "USTRF Sep30'25 Smartphones YES @FORECASTX",
  'details': ''},
 'kind': 'discrete',
 'bins': [{'props': {'title': 'The United States will impose additional tariffs on smartphones by September 30, 2025.',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'The United States will not impose additional tariffs on smartphones by September 30, 2025.',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0}]}

In [262]:
contracts = df.to_dict(orient='records')

In [266]:
contracts = {contract['conid']: contract for contract in contracts}

In [ ]:
for ifp in ifps:
    fn = f'glimt/forecast'
    fn = f"{fn}/{ifp['id']}.json"
    with open(fn, 'r') as f:
        forecast = json.load(f)
    [[Pyes, Pno], Rfor, Ragainst, sources] = forecast
    contract = contracts[ifp['id']]
    yesNo = contract['yesNo']
    pct = contract['probability_pct']
    yesPct = pct if yesNo == 'YES' else 100-pct
    noPct = 100-pct if yesNo == 'YES' else pct
    report = f"""
{ifp['props']['title']}
{ifp['props']['shortTitle']}

YES {100*Pyes} bot / {yesPct} crowd
NO  {100*Pno} bot  / {noPct} crowd

Reasons for YES
===============
{Rfor}

Reasosn for NO
==============
{Ragainst}

----------------------------------------"""
    print(report)